<a href="https://colab.research.google.com/github/danielandresrestrebadaleal-dev/ARC/blob/main/corner_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Overview

This code analyzes how different launch parameters (like launch latitude, exhaust velocity, etc.) affect the required delta-V and propellant mass to insert a payload in a geosynchronous orbit with a desired inclination. It creates a corner plots to understand the dependency of the delta-V and fuel mass on these launch variables

In [ ]:
#Install necessary packages for Google Colab. Note: hapsira is the modern version of poliastro to handle Python 3.12
!pip install hapsira "astropy<7.0" corner

# Code Imports
This code utilizes several Python libraries for different functionalities:

1. poliastro & astropy: for orbital state representation

2. multiprocessing: to handle multiple orbital simulations simultaneously to reduce computational time

3. corner & matplotlib: to plot results

4. random: to randomize some launch parameters

5. time: to keep track of the computational time

6. numpy: to make complex vector calculations

7. simulate_orbit: function that simulates the insertion of a payload into GEO orbit and computes the total delta-V and propellant mass needed given specific launch parameters (from GEO_orbital_simulation tutorial)

In [ ]:
import numpy as np
import time
import random
from itertools import product
import multiprocessing as mp
import matplotlib as mpl
from matplotlib import pyplot as plt
import corner

from astropy import units as u
from hapsira.bodies import Earth
from hapsira.twobody import Orbit
from astropy.coordinates import CartesianRepresentation

# Import the simulation function from the simulation module.
# Ensure that the function 'simulate' is available in the 'simulation.py' module in your environment.
from simulation import simulate_orbit

# Defining launch parameters

All the launch parameters that will be analyzed are initialized here. Discrete values are chosen for all quantities and are randomized within a range except the final inclination and exhaust velocities of the two stages of the rocket.

In [7]:
# ===============================================================
# Define Parameter Lists for Simulation
# ===============================================================
# Fixed list of inclination values (in degrees)
inclination_values = [5, 13, 26, 34, 46, 62]

# Fixed list of stage 1 exhaust velocities (in m/s)
exhaust_velocity_stage1 = [2800, 2925, 3050, 3175, 3300]

# Fixed list of stage 2 exhaust velocities (in m/s)
exhaust_velocity_stage2 = [4200, 4300, 4400, 4500, 4600]

# Generate randomized list for payload exhaust velocity (propulsive exhaust) in m/s
payload_exhaust_lin = np.linspace(20000, 30000, 5)
payload_exhaust_values = [
    np.clip(val + np.random.uniform(-1000, 1000), 20000, 30000)
    for val in payload_exhaust_lin
]

# Generate randomized list for LEO altitude values (in km)
leo_altitude_lin = np.linspace(200, 800, 10)
leo_altitude_values = [
    np.clip(val + np.random.uniform(-10, 10), 200, 800)
    for val in leo_altitude_lin
]

# Generate randomized list for payload mass values (in kg)
payload_mass_lin = np.linspace(200, 300, 5)
payload_mass_values = [
    np.clip(val + np.random.uniform(-10, 10), 200, 300)
    for val in payload_mass_lin
]

# Generate randomized list for inclination change percentage (in %)
inclination_change_lin = np.linspace(0, 100, 10)
inclination_change_values = [
    np.clip(val + np.random.uniform(-5, 5), 0, 100)
    for val in inclination_change_lin
]

# Main code

Here, we combine all the launch parameters into a tuple and pass them over to the simulation_orbit() function. The multiprocessing tool is utilized to run multiple simulations with different parameters at the same time to run the code faster. Lastly, we create a corner plot using the corner function and adjust font size, dpi of the picture, and other variabes to create a polished figure with the help of matplotlib

In [ ]:
# ===============================================================
# Main Entry Point for Simulation and Plotting
# ===============================================================
if __name__ == '__main__':
    # Mark start time of simulation
    start_time = time.perf_counter()

    # Create a list of all possible parameter combinations as tuples:
    # (inclination, LEO altitude, stage 1 exhaust, stage 2 exhaust, payload exhaust, payload mass, inclination change)
    parameter_combinations = list(product(
        inclination_values,
        leo_altitude_values,
        exhaust_velocity_stage1,
        exhaust_velocity_stage2,
        payload_exhaust_values,
        payload_mass_values,
        inclination_change_values
    ))

    # Use a multiprocessing pool to parallelize the simulation.
    with mp.Pool(processes=mp.cpu_count()) as pool:
        simulation_results = pool.map(simulate_orbit, parameter_combinations)

    # Convert the results to a NumPy array for further processing/plotting.
    results_array = np.array(simulation_results)

    # Set global font size for plotting with matplotlib
    mpl.rcParams.update({'font.size': 115})

    # Define axis labels for the corner plot
    plot_labels = [
        "Initial Inclination (deg)",
        "Inclination Change (%)",
        "LEO Altitude (km)",
        "Stage 1 EV (m/s)",
        "Stage 2 EV (m/s)",
        "Payload EV (m/s)",
        "Payload Mass (kg)",
        "Propellant Mass (kg)",
        "Total ΔV (m/s)"
    ]

    # Generate the corner plot with the simulation results
    corner_fig = corner.corner(
        results_array,
        labels=plot_labels,
        label_kwargs={"fontsize": 120},
        show_titles=True,
        title_fmt=".2f",
        bins=60,
        smooth=1.0,
        fill_contours=True,
        plot_contours=True,
        max_n_ticks=4,
        hist2d_kwargs={
            'plot_datapoints': True,
            'alpha': 0.15,
            'plot_density': False
        },
        scatter_kwargs={
            'alpha': 0.05,
            's': 3
        }
    )

    # Set figure size and layout adjustments, then save and display the plot.
    corner_fig.set_size_inches(220, 220)
    corner_fig.tight_layout()
    plt.savefig("Corner.pdf", dpi=130, bbox_inches='tight')

    # Mark end time and output the duration of the simulation
    end_time = time.perf_counter()
    print("Total simulation and plotting time: ", end_time - start_time, "seconds")
    plt.show()

Given the size of this figure, the code below is used to download the figure from a google colab folder and transfer it to your google drive account for visualization with the name of "Corner_Simulation_Plot.pdf." Please run the code below to download the figure


In [ ]:
from google.colab import drive
import shutil

# 1. This will prompt you to log in and connect your Google Drive
drive.mount('/content/drive')

# 2. This copies your massive PDF from Colab's temporary memory straight into your Drive
shutil.copy("Corner.pdf", "/content/drive/MyDrive/Corner_Simulation_Plot.pdf")

print("Transfer complete! Check your Google Drive.")

# Results and Conclusions

As shown in the corner plot above, the three main factors to consider to otpimize the payload mass are the launch site latitude, the exhaust velocity of the stage 2 of the rocket, and the mass of the payload. Similarly, the percent change in final inclination and initial launch site latitude are the factors that most heavily influence the total required delta-V. These results highlight the need for a strategic design of the payload to reduce the fuel mass, as well as selecting high-performance engines, especially for the stage 2 of the rocket. Additionally, the initial and final inclinations of the orbit must be close in value if minimizing the delta-V is a priority.